# COMP5328 Assignment 2 — learning notebook

This notebook is a guided scaffold, not a completed solution. Work from top to bottom. Each important assignment operation is left as a `TODO`, followed by a hint and a checkpoint.

The central picture is:

**image → CNN logits → clean probabilities `p` → transition matrix `T` → noisy probabilities `q`**

Training labels are noisy (`Str`). Clean test labels (`Yts`) are used only after training and model selection are finished.

## Roadmap

1. Inspect one dataset without changing it.
2. Build a reproducible 80/20 split.
3. Convert images into PyTorch's expected layout.
4. Build a small CNN with three outputs.
5. Verify the transition-matrix orientation.
6. Implement forward correction.
7. Train a noisy-label pilot and estimate `T0`.
8. Implement the paper-specific reweighted loss.
9. Add the CIFAR matrix-revision stage.
10. Run paired experiments and summarize clean-test accuracy.

Do not move to the next stage until the current checkpoint passes.

In [ ]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

print('PyTorch:', torch.__version__)
print('MPS available:', torch.backends.mps.is_available())
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# If Jupyter starts in the repository root, this is correct. Change it if needed.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'

DATASETS = {
    'fashion_03': {
        'filename': 'FashionMNIST0.3.npz',
        'train_count': 18_000,
        'test_count': 3_000,
        'image_shape': (28, 28),
        'channels': 1,
    },
    'fashion_06': {
        'filename': 'FashionMNIST0.6.npz',
        'train_count': 18_000,
        'test_count': 3_000,
        'image_shape': (28, 28),
        'channels': 1,
    },
    'cifar': {
        'filename': 'CIFAR.npz',
        'train_count': 15_000,
        'test_count': 3_000,
        'image_shape': (32, 32, 3),
        'channels': 3,
    },
}

def choose_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = choose_device()
print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)

## 1. Inspect the raw arrays

Goal: learn what is actually stored before transforming anything. The four required keys are `Xtr`, `Str`, `Xts`, and `Yts`. Labels must be integers from 0 to 2.

**Important:** do not automatically divide images by 255. First inspect the pixel minimum and maximum.

In [ ]:
REQUIRED_KEYS = {'Xtr', 'Str', 'Xts', 'Yts'}

def inspect_npz(path, expected):
    """Inspect and validate one supplied dataset without modifying it."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Place the supplied dataset at {path}')

    with np.load(path) as archive:
        available_keys = set(archive.files)
        print('Keys:', sorted(available_keys))

        # TODO 1: check that REQUIRED_KEYS are present.
        # Hint: subtract available_keys from REQUIRED_KEYS to find missing keys.

        arrays = {name: archive[name] for name in REQUIRED_KEYS}

    for name, array in arrays.items():
        print(name, 'shape=', array.shape, 'dtype=', array.dtype)

    # TODO 2: check the expected training and test counts.
    # TODO 3: check the expected image shapes.
    # TODO 4: check that Str and Yts contain only 0, 1, and 2.
    # TODO 5: print image minima and maxima for Xtr and Xts.

    return arrays

# Start with ONE dataset only. Uncomment after copying the supplied file into data/.
# selected = DATASETS['fashion_03']
# arrays = inspect_npz(DATA_DIR / selected['filename'], selected)

### Checkpoint 1

Before continuing, you should be able to explain:

- The shape and range of every array.
- Why `Str` is allowed during training but `Yts` is not.
- Whether scaling by 255 is needed.

Send me the printed output and your completed `inspect_npz` function for checking.

## 2. Reproducible train/validation split

Split `Xtr`/`Str` into 80% training and 20% validation. Work with indices so the exact split can be saved and reused by both methods.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_split(num_examples, seed, validation_fraction=0.20):
    """Return disjoint, reproducible training and validation index arrays."""
    # TODO 1: create a local NumPy random generator from seed.
    # TODO 2: create a random permutation of all example indices.
    # TODO 3: calculate the validation size.
    # TODO 4: slice the permutation into validation and training indices.
    # TODO 5: assert that there is no overlap and all examples are included.
    raise NotImplementedError

# Suggested checks after implementation:
# train_a, val_a = make_split(100, seed=7)
# train_b, val_b = make_split(100, seed=7)
# train_c, val_c = make_split(100, seed=8)
# assert np.array_equal(train_a, train_b)
# assert not np.array_equal(train_a, train_c)

## 3. Image conversion and PyTorch dataset

PyTorch convolution layers expect images in channel-first form: `[channels, height, width]`. In the supplied files, Fashion is flattened as `[N, 784]`, so reshape it to `[N, 1, 28, 28]`. CIFAR is `[N, 32, 32, 3]`, so move its last dimension to obtain `[N, 3, 32, 32]`. The images are `uint8` in the range 0–255, so convert to `float32` and divide by 255.

In [ ]:
def prepare_images(images):
    """Return float32 channel-first images without guessing the scale."""
    # TODO 1: convert to float32.
    # TODO 2: reshape Fashion [N, 784] to [N, 1, 28, 28].
    # TODO 3: move CIFAR [N, 32, 32, 3] to [N, 3, 32, 32].
    # TODO 4: divide by 255 because inspection showed uint8 pixels in [0, 255].
    # TODO 5: assert the result has shape [N, C, H, W].
    raise NotImplementedError

class IndexedNoisyDataset(Dataset):
    def __init__(self, images, noisy_labels, indices):
        self.images = images
        self.noisy_labels = noisy_labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        # TODO: use self.indices[position] to select the matching image and label.
        # Return a float image tensor and a long-integer label tensor.
        raise NotImplementedError

## 4. Small CNN

The CNN must return exactly three logits. Do not place softmax inside the model; different losses and evaluation steps need access to the raw logits. Use the same backbone for both methods on a given dataset.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, input_channels, num_classes=3):
        super().__init__()
        # TODO: create a modest feature extractor.
        # Hint: repeat convolution -> activation -> pooling two or three times.
        self.features = None

        # TODO: create a classifier ending in num_classes logits.
        # Adaptive pooling can avoid manually calculating flattened dimensions.
        self.classifier = None

    def forward(self, x):
        # TODO: features -> classifier -> logits
        raise NotImplementedError

# After implementation, test both channel counts.
# assert SmallCNN(1)(torch.zeros(4, 1, 28, 28)).shape == (4, 3)
# assert SmallCNN(3)(torch.zeros(4, 3, 32, 32)).shape == (4, 3)

## 5. Transition matrices and orientation

This project uses `T[i, j] = P(observed label j | true label i)`. Rows are clean classes and columns are noisy labels. For a batch, noisy probabilities are `q = p @ T`.

In [ ]:
T_FASHION_03 = torch.tensor([
    [0.7, 0.3, 0.0],
    [0.0, 0.7, 0.3],
    [0.3, 0.0, 0.7],
], dtype=torch.float32)

T_FASHION_06 = torch.tensor([
    [0.4, 0.3, 0.3],
    [0.3, 0.4, 0.3],
    [0.3, 0.3, 0.4],
], dtype=torch.float32)

def clean_to_noisy_probabilities(clean_probabilities, transition_matrix):
    # TODO: apply the stated matrix convention. This should be one matrix operation.
    raise NotImplementedError

def check_transition_matrix(transition_matrix, atol=1e-6):
    # TODO: assert shape [3, 3], finite values, nonnegative entries, and row sums of one.
    raise NotImplementedError

# Orientation check: a certain clean class 0 must become row 0 of T.
# p_one_hot = torch.tensor([[1.0, 0.0, 0.0]])
# q = clean_to_noisy_probabilities(p_one_hot, T_FASHION_03)
# assert torch.allclose(q[0], T_FASHION_03[0])

## 6. Forward-correction loss

Algorithm: logits → softmax → `p` → `p @ T` → `q` → select each observed label's probability → negative log → mean. Do not pass `q` to ordinary cross-entropy because `q` is already a probability distribution.

In [ ]:
def forward_correction_loss(logits, noisy_labels, transition_matrix, epsilon=1e-8):
    # TODO 1: convert logits into clean-class probabilities p.
    # TODO 2: calculate noisy-label probabilities q.
    # TODO 3: select q[b, noisy_labels[b]] for every batch item.
    # Hint: torch.gather is useful. Pay attention to dimensions.
    # TODO 4: protect against log(0), take negative logs, then average.
    raise NotImplementedError

# Identity-matrix check after implementation:
# logits = torch.randn(8, 3)
# labels = torch.randint(0, 3, (8,))
# robust = forward_correction_loss(logits, labels, torch.eye(3))
# ordinary = nn.functional.cross_entropy(logits, labels)
# assert torch.allclose(robust, ordinary, atol=1e-6)

## 7. Shared training and noisy-validation flow

Training updates parameters. Validation does not. A noise-aware model is validated by comparing `argmax(q)` with noisy validation labels, not by comparing `argmax(p)` with them.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_function, device):
    model.train()
    # TODO: for each batch, move data, clear gradients, calculate logits and loss,
    # backpropagate, update parameters, and accumulate a sample-weighted mean loss.
    raise NotImplementedError

@torch.no_grad()
def validate_noise_aware(model, loader, transition_matrix, device, epsilon=1e-8):
    model.eval()
    # TODO: calculate p, then q, noisy-label error, and noisy-label NLL.
    # Return both metrics so error is primary and NLL breaks ties.
    raise NotImplementedError

def is_better_checkpoint(candidate, best):
    # TODO: lower noisy validation error wins; if tied, lower NLL wins.
    raise NotImplementedError

## 8. Pilot classifier and `T0` estimation

The pilot predicts noisy labels directly. Select its checkpoint on noisy validation data. Then run that checkpoint over training images only. For each class `i`, find the training image with the largest predicted probability for `i`; use that image's whole probability vector as row `i` of `T0`.

In [ ]:
@torch.no_grad()
def estimate_transition_matrix(pilot_model, training_loader, device, num_classes=3):
    pilot_model.eval()
    # TODO 1: collect the pilot softmax vector for every training example.
    # TODO 2: for each class i, find the example maximizing probability column i.
    # TODO 3: copy that example's entire vector into row i of T0.
    # TODO 4: validate the resulting matrix.
    raise NotImplementedError

# Important questions:
# - Does training_loader cover only the training portion?
# - Is shuffle=False useful here for traceability?
# - Why is this only an approximate estimate?

## 9. Paper-specific reweighted loss

For observed label `s`, use `weight = p[s] / q[s]`. Multiply that weight by ordinary per-example cross-entropy between the clean logits and `s`, then average.

Leave gradient detachment and clipping as explicit choices. We will verify them against the paper and reference implementation before the final experiment.

In [ ]:
def reweighted_loss(
    logits,
    noisy_labels,
    transition_matrix,
    epsilon=1e-8,
    detach_weights=False,
    max_weight=None,
):
    # TODO 1: calculate clean probabilities p and noisy probabilities q.
    # TODO 2: gather p[s] and q[s] for every example.
    # TODO 3: form the protected ratio p[s] / q[s].
    # TODO 4: optionally detach and/or clip using the named arguments.
    # TODO 5: calculate per-example cross-entropy with reduction='none'.
    # TODO 6: return the mean of weight * per-example cross-entropy.
    raise NotImplementedError

# Identity-matrix check: weights should become one, so this should match ordinary CE.

## 10. CIFAR T-Revision stage

First estimate `T0`. Next initialize the classifier using fixed-`T0` reweighting. Finally learn the classifier and a matrix revision together.

Your chosen matrix parameterization must guarantee nonnegative entries and rows that sum to one. If it differs from the paper's reference implementation, describe it as an adaptation rather than claiming an exact reproduction.

In [ ]:
class RevisedTransition(nn.Module):
    def __init__(self, initial_transition):
        super().__init__()
        self.register_buffer('T0', initial_transition.clone())
        # TODO: create revision parameters initialized so the starting matrix equals T0.

    def forward(self):
        # TODO: combine T0 and revision parameters.
        # TODO: enforce a valid row-stochastic matrix.
        raise NotImplementedError

# Required checks:
# 1. The initial returned matrix matches T0.
# 2. Every returned row sums to one.
# 3. Entries are nonnegative.
# 4. Revision parameters receive finite gradients.

## 11. Clean test evaluation

Only after training, tuning, early stopping, and checkpoint selection are complete may you use `Xts` and `Yts`. Test predictions come from `argmax(p)` before applying the transition matrix.

In [ ]:
@torch.no_grad()
def evaluate_clean_test(model, test_loader, device):
    model.eval()
    # TODO: calculate logits, choose argmax directly from clean outputs,
    # compare with Yts, and return accuracy as a percentage.
    # Do not accept a transition matrix in this function.
    raise NotImplementedError

## 12. Paired experiment plan

For every dataset and seed:

1. Create one 80/20 split.
2. Give the same split to both methods.
3. Fit any preprocessing on training data only.
4. Estimate `T0` where required.
5. Train and select both methods independently.
6. Evaluate selected checkpoints on clean test data.
7. Save accuracy, runtime, stopping epoch, configuration, and matrices.
8. Save the paired accuracy difference.

Use one seed and short epoch limits for smoke tests. Use at least ten distinct seeds for final results. Never mix smoke results into final summaries.

In [ ]:
def summarize_results(results_frame):
    # TODO: group by dataset and method.
    # Report count, mean accuracy, and sample SD (ddof=1).
    # Add a flag saying whether at least 10 final runs were completed.
    raise NotImplementedError

# Expected long-form columns could include:
RESULT_COLUMNS = [
    'dataset', 'method', 'seed', 'smoke', 'test_accuracy',
    'runtime_seconds', 'selected_epoch', 'checkpoint_path',
]

## Recommended learning stops

Do not try to fill the whole notebook at once. Send it for checking after each group:

- **Stop A:** Sections 1–3: loading, splitting, and image conversion.
- **Stop B:** Sections 4–6: CNN, matrix convention, and forward loss.
- **Stop C:** Sections 7–8: training loop, validation, pilot, and `T0`.
- **Stop D:** Section 9: reweighted loss.
- **Stop E:** Sections 10–11: revision and clean-test evaluation.
- **Stop F:** Section 12: smoke run and paired experiment pipeline.

At every stop, I can run the notebook, inspect outputs, and explain failures without replacing your work with a finished solution.